# Full Day Antenna Flagging

**by Josh Dillon**, last updated August 27, 2026

This notebook is designed to harmonize the potentially inconsistent per-antenna flagging coming out of the [file_sky_calibration](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/file_sky_calibration.ipynb) notebook. It seeks to flag likely bad times between known bad times and to impose maximum flagging fractions and maximum stretches of consecutive flags between otherwise good data (which can raise problems when smoothing or filtering later). It also verifies that any antenna identity relabelings are consistent across the whole day, flagging any antenna whose identity shows temporal structure (a mid-night relabel change cannot be applied coherently downstream).

Here's a set of links to skip to particular figures and tables:

• [Relabeling Consistency Report](#Check-relabeling-consistency)

• [Figure 1: Flag Summary vs. JD](#Figure-1:-Flag-Summary-vs.-JD)

• [Figure 2: Array Flag Fraction Summary](#Figure-2:-Array-Flag-Fraction-Summary)

• [Figure 3: Flag Fraction vs. JD Summary](#Figure-3:-Flag-Fraction-vs.-JD-Summary)

• [Figure 4: Per-Antenna Flag Harmonization Summary](#Figure-4:-Per-Antenna-Flag-Harmonization-Summary)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import json
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import pandas as pd
import glob
import copy
import matplotlib
import matplotlib.pyplot as plt
from pyuvdata import UVFlag, UVData, UVCal
from hera_cal import io, utils
from hera_qm.time_series_metrics import true_stretches, impose_max_flag_gap, metric_convolution_flagging
from hera_qm.metrics_io import read_a_priori_int_flags
from uvtools.plot import plot_antpos, plot_antclass
from scipy.ndimage import convolve
from astropy.io import fits
%matplotlib inline
from IPython.display import display, HTML

## Parse inputs

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"

# default file suffixes, overridden by the TOML's [DATA_PRODUCTS] section (if given), so
# that no filename convention is duplicated across the notebooks and the task scripts
SUM_SUFFIX = 'sum.uvh5'
SKY_CAL_SUFFIX = 'sum.sky.calfits'
ANT_CLASS_SUFFIX = 'sum.ant_class.csv'
ANTENNA_FLAGS_SUFFIX = 'sum.antenna_flags.h5'

# other defaults, overridden by the TOML's [FULL_DAY_ANT_FLAG_OPTS] section (if given)
APRIORI_YAML_PATH = None
SKIP_OUTRIGGERS = True

# parameters for harmonizing partially-flagged antennas
SMOOTHING_SCALE_NFILES = 30
MAX_FLAG_GAP_NFILES = 30

# max flag fractions (before just flagging the whole antenna)
AUTO_POWER_MAX_FLAG_FRAC = 0.5
AUTO_SHAPE_MAX_FLAG_FRAC = 0.25
AUTO_SLOPE_MAX_FLAG_FRAC = 0.25
AUTO_RFI_MAX_FLAG_FRAC = 0.25
CHISQ_MAX_FLAG_FRAC = 0.5
CAL_AUTO_MAX_FLAG_FRAC = 0.5
DECO_MAX_FLAG_FRAC = 0.5
IDENTITY_MAX_FLAG_FRAC = 0.25
XENGINE_MAX_FLAG_FRAC = 0.05
OVERALL_MAX_FLAG_FRAC = 0.5

# load TOML overrides, injecting each key as an uppercase global
# which [WorkFlow] action is running this notebook. The task script exports it (it is
# named do_$ACTION.sh), so a template shared by two actions still identifies itself;
# the fallback is only for running this notebook by hand.
ACTION = os.environ.get("ACTION", "FULL_DAY_ANTENNA_FLAGGING_NOTEBOOK")
toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
if 'FULL_DAY_ANT_FLAG_OPTS' in toml_options:
    print(f'Loading overrides from [FULL_DAY_ANT_FLAG_OPTS] in {TOML_FILE}.')
    for key, val in toml_options['FULL_DAY_ANT_FLAG_OPTS'].items():
        globals()[key.upper()] = val

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'SUM_SUFFIX', 'SKY_CAL_SUFFIX', 'ANT_CLASS_SUFFIX',
                'ANTENNA_FLAGS_SUFFIX', 'APRIORI_YAML_PATH', 'SKIP_OUTRIGGERS', 'SMOOTHING_SCALE_NFILES',
                'MAX_FLAG_GAP_NFILES', 'AUTO_POWER_MAX_FLAG_FRAC', 'AUTO_SHAPE_MAX_FLAG_FRAC',
                'AUTO_SLOPE_MAX_FLAG_FRAC', 'AUTO_RFI_MAX_FLAG_FRAC', 'CHISQ_MAX_FLAG_FRAC', 'CAL_AUTO_MAX_FLAG_FRAC',
                'DECO_MAX_FLAG_FRAC', 'IDENTITY_MAX_FLAG_FRAC', 'XENGINE_MAX_FLAG_FRAC',
                'OVERALL_MAX_FLAG_FRAC']:
    print(f'{setting} = {eval(setting)}')

In [ ]:
# Default classification bound scalars, overridden by the TOML's [ANT_CLASS_BOUNDS] section (if given).
# These are the same section and defaults the file_sky_calibration notebook uses, so the day-level
# bounds cannot drift from the per-file ones.
AM_CORR_BAD = 0.35
AM_XPOL_BAD = -0.1
BAD_SOLAR_ALTITUDE = -2.0
AUTO_POWER_SUSPECT_LOW = 1.0
AUTO_POWER_SUSPECT_HIGH = 60.0
AUTO_SLOPE_GOOD_LOW = -0.4
AUTO_SLOPE_GOOD_HIGH = 0.4
AUTO_RFI_GOOD = 1.5
AUTO_SHAPE_GOOD = 0.1
SC_CSPA_SUSPECT = 3.0
CAL_AUTO_MODEL_SUSPECT = 2.0
DECO_MAX_BAD = 0.10
DECO_JUMP_BAD = 0.05
IDENTITY_COHERENCE_SUSPECT = 0.5

if 'ANT_CLASS_BOUNDS' in toml_options:
    print(f'Loading overrides from [ANT_CLASS_BOUNDS] in {TOML_FILE}.')
    for key, val in toml_options['ANT_CLASS_BOUNDS'].items():
        globals()[key.upper()] = val

# bounds used by the flagging steps below
am_corr_bad = (0, AM_CORR_BAD)
am_xpol_bad = (-1, AM_XPOL_BAD)
auto_power_suspect = (AUTO_POWER_SUSPECT_LOW, AUTO_POWER_SUSPECT_HIGH)
auto_slope_good = (AUTO_SLOPE_GOOD_LOW, AUTO_SLOPE_GOOD_HIGH)
auto_rfi_good = (0, AUTO_RFI_GOOD)
auto_shape_good = (0, AUTO_SHAPE_GOOD)
sc_cspa_suspect = (0, SC_CSPA_SUSPECT)
cal_auto_model_suspect = (1 / CAL_AUTO_MODEL_SUSPECT, CAL_AUTO_MODEL_SUSPECT)

for bound in ['am_corr_bad', 'am_xpol_bad', 'BAD_SOLAR_ALTITUDE', 'auto_power_suspect', 'auto_slope_good',
              'auto_rfi_good', 'auto_shape_good', 'sc_cspa_suspect', 'cal_auto_model_suspect', 'DECO_MAX_BAD', 'DECO_JUMP_BAD',
              'IDENTITY_COHERENCE_SUSPECT']:
    print(f'{bound} = {eval(bound)}')

## Load data

In [ ]:
hd = io.HERAData(SUM_FILE)

In [ ]:
sum_glob = '.'.join(SUM_FILE.split('.')[:-3]) + '.*.' + SUM_SUFFIX
cal_files_glob = sum_glob.replace(SUM_SUFFIX, SKY_CAL_SUFFIX)
cal_files = sorted(glob.glob(cal_files_glob))
print(f'Found {len(cal_files)} *.{SKY_CAL_SUFFIX} files starting with {cal_files[0]}.')

In [ ]:
ant_class_csvs_glob = sum_glob.replace(SUM_SUFFIX, ANT_CLASS_SUFFIX)
ant_class_csvs = sorted(glob.glob(ant_class_csvs_glob))
jds = [float(f.split('/')[-1].split('zen.')[-1].split('.sum')[0]) for f in ant_class_csvs]
frac_jds = jds - np.floor(jds[0])
lst_hours = utils.JD2LST(np.array(jds)) * 12 / np.pi
lst_hours[lst_hours > lst_hours[-1]] -= 24
Ncsvs = len(ant_class_csvs)
print(f'Found {Ncsvs} *.{ANT_CLASS_SUFFIX} files starting with {ant_class_csvs[0]}.')

In [ ]:
# Load ant_class csvs, aligning rows on the union of antennas: files can differ in which
# antennas they contain (e.g. correlator dropouts), and a missing antenna is treated as
# bad in that file (blank metrics, 'bad' classifications)
tables = [pd.read_csv(f).dropna(axis=0, how='all') for f in ant_class_csvs]
table_cols = tables[0].columns[1::2]
class_cols = tables[0].columns[2::2]
ap_strs = np.array(sorted({ap for t in tables for ap in t['Antenna']}, key=lambda ap: (int(ap[:-1]), ap[-1])))
ants = sorted(set(int(a[:-1]) for a in ap_strs))
per_file_ap_sets = [set(t['Antenna']) for t in tables]
missing = {ap: sum(ap not in aps for aps in per_file_ap_sets) for ap in ap_strs
           if any(ap not in aps for aps in per_file_ap_sets)}
if len(missing) > 0:
    print('Antpols missing from some files (and treated as bad there): '
          + ', '.join(f'{ap} ({nfiles} files)' for ap, nfiles in missing.items()))
tables = [t.set_index('Antenna').reindex(ap_strs) for t in tables]
for t in tables:
    t[list(class_cols)] = t[list(class_cols)].fillna('bad')

In [ ]:
# build up dictionaries for full night metrics and classifications
replace = {'-': np.nan, 'INF': np.inf, 'No': False, 'Yes': True}


def parse_metric(val):
    '''Converts a CSV metric entry to a float: blanks (nan) pass through, percentages
    become fractions, and the replace dict handles the special strings.'''
    if isinstance(val, str):
        if val in replace:
            return replace[val]
        if val.endswith('%'):
            return float(val[:-1]) / 100
    return float(val)


metric_data = {tc: {} for tc in table_cols}
class_data = {tc: {} for tc in table_cols}
for tc, cc in zip(table_cols, class_cols):
    class_array = np.vstack([t[cc] for t in tables]).T
    metric_array = np.vstack([(t.index if tc == 'Antenna' else t[tc]) for t in tables]).T
    for ca, ma, ap_str in zip(class_array, metric_array, ap_strs):
        class_data[tc][ap_str] = ca
        if tc == 'Antenna':
            metric_data[tc][ap_str] = ma
        elif tc == 'Dead?':
            # keep this a clean boolean: an antenna missing from a file (nan) is flagged
            # through its 'bad' classifications, not through the Dead? metric
            metric_data[tc][ap_str] = np.array([val == 'Yes' for val in ma])
        else:
            metric_data[tc][ap_str] = np.array([parse_metric(val) for val in ma])

In [ ]:
original_flags = {ap_str: np.array(class_data['Antenna'][ap_str] == 'bad') for ap_str in ap_strs}
print(f'{np.mean(list(original_flags.values())):.2%} of antenna-files flagged by the per-file notebook.')

## Check relabeling consistency

The identity audit in `file_sky_calibration` can relabel antennas whose visibilities demonstrably belong to a different antenna, recording each file's repair map in its calfits `RELABELS` header keyword. Downstream consumers of raw data apply each file's own map, which is only well-defined if every antenna is relabeled *the same way* in every file where it is active (files where an antenna is entirely flagged are exempt, since the audit may not have run or repaired there). An antenna whose identity shows temporal structure — a flickering or mid-night-changing relabel — cannot be used coherently, so it is **flagged for the entire day** and reported loudly here.

In [ ]:
# read each file's identity-repair record: the RELABELS JSON in the calfits header
relabel_maps = []
for cal in cal_files:
    header = fits.getheader(cal)
    relabel_maps.append({int(labeled): int(true) for labeled, true in json.loads(header['RELABELS']).items()}
                        if 'RELABELS' in header else {})

# for each antenna involved in any relabel (as either label or true identity), collect its mapping
# "signature" (who it is relabeled to, and who claims to be it) in every file where it is active
# (i.e. where any of its antpols is unflagged)
inconsistent, consistent = {}, {}
for antnum in sorted({a for rl in relabel_maps for pair in rl.items() for a in pair}):
    active = ~np.all([original_flags[ap_str] for ap_str in ap_strs if int(ap_str[:-1]) == antnum], axis=0)
    signatures = {}
    for i, rl in enumerate(relabel_maps):
        if active[i]:
            sig = (rl.get(antnum), tuple(sorted(labeled for labeled, true in rl.items() if true == antnum)))
            signatures.setdefault(sig, []).append(i)
    (inconsistent if len(signatures) > 1 else consistent)[antnum] = signatures


def describe(as_label, claimed_by):
    desc = (f'is relabeled to antenna {as_label}' if as_label is not None else 'keeps its own label')
    if len(claimed_by) > 0:
        desc += f' and is claimed by antenna{"s" if len(claimed_by) > 1 else ""} {", ".join(map(str, claimed_by))}'
    return desc


for antnum, signatures in consistent.items():
    for (as_label, claimed_by), idxs in signatures.items():
        print(f'Antenna {antnum} consistently {describe(as_label, claimed_by)} in all {len(idxs)} files where it is active.')
if len(inconsistent) > 0:
    print('!' * 110)
    print('!!! INCONSISTENT RELABELING DETECTED: these antennas will be flagged for the ENTIRE DAY (see STEP 4 below).')
    for antnum, signatures in inconsistent.items():
        print(f'!!!\n!!! Antenna {antnum}:')
        for (as_label, claimed_by), idxs in sorted(signatures.items(), key=lambda kv: -len(kv[1])):
            print(f'!!!     {describe(as_label, claimed_by)} in {len(idxs)} files '
                  f'(JD {jds[min(idxs)]:.5f} to {jds[max(idxs)]:.5f})')
    print('!' * 110)
else:
    print('No relabeling inconsistencies detected across the day.')

In [ ]:
class FlagHistory:
    '''Helps keep strack of why flags get applied'''
    
    def __init__(self, ap_strs, Nfiles):
        self.final_flags = {ap_str: np.zeros(Nfiles, dtype=bool) for ap_str in ap_strs}
        self.history = {}
    
    def update(self, ap_str, per_file_flags, updated_flags, rationale):
        self.history[(ap_str, rationale)] = (np.array(self.final_flags[ap_str]), np.array(per_file_flags), np.array(updated_flags))
        self.final_flags[ap_str] |= updated_flags
        
    def summarize(self, description):
        print(f'{np.mean(list(self.final_flags.values())):.2%} of antenna-files flagged after {description}.')


def nan_fill(metric):
    '''Replaces nan entries (e.g. unmeasured files) with the Gaussian-weighted average of their
    finite neighbors, so that smoothing-based flag growth works through short unmeasured gaps
    between bad files while unmeasured files with good neighbors are certified by those
    neighbors. Returns None if nothing is finite.'''
    finite = np.isfinite(metric)
    if not np.any(finite):
        return None
    if np.all(finite):
        return np.asarray(metric, dtype=float)
    kernel = np.exp(-np.arange(-len(metric) // 2, len(metric) // 2 + 1)**2 / 2 / SMOOTHING_SCALE_NFILES**2)
    with np.errstate(invalid='ignore'):
        filled = np.where(finite, metric, convolve(np.where(finite, metric, 0), kernel, mode='reflect')
                          / convolve(finite.astype(float), kernel, mode='reflect'))
    # gaps so wide the kernel underflows to zero weight fall back to the finite mean
    return np.where(np.isfinite(filled), filled, np.mean(metric[finite]))


def nan_aware_smooth(metric):
    '''The Gaussian-smoothed nan_fill of a metric (all-nan if nothing is finite): exactly the
    smoothed metric that grow_flags_between_bad_times tests against ok_range.'''
    filled = nan_fill(metric)
    if filled is None:
        return np.full(len(metric), np.nan)
    kernel = np.exp(-np.arange(-len(metric) // 2, len(metric) // 2 + 1)**2 / 2 / SMOOTHING_SCALE_NFILES**2)
    return convolve(filled, kernel / np.sum(kernel), mode='reflect')


def grow_flags_between_bad_times(metric, starting_flags, ok_range):
    '''metric_convolution_flagging on the nan_fill'd metric: flags grow to cover stretches between
    flagged files where the smoothed metric never returns to ok_range.'''
    filled = nan_fill(metric)
    if filled is None:
        return np.array(starting_flags)
    return metric_convolution_flagging(filled, starting_flags, ok_range,
                                       sigma=SMOOTHING_SCALE_NFILES, max_flag_gap=MAX_FLAG_GAP_NFILES)

## Perform flagging

In [ ]:
fh = FlagHistory(ap_strs, Ncsvs)

# STEP 1: FLAG SUN-UP DATA AND OTHER A PRIORI FLAGGED TIMES
for ap_str in ap_strs:
    sun_up = metric_data['Solar Alt'][ap_str] > BAD_SOLAR_ALTITUDE
    fh.update(ap_str, sun_up, sun_up, 'Solar Alt')
solar_flags = np.all([metric_data['Solar Alt'][ap_str] > BAD_SOLAR_ALTITUDE for ap_str in ap_strs], axis=0)
fh.summarize('flagging sun-up data')
if APRIORI_YAML_PATH is not None:
    apriori_flags = np.zeros(len(jds), dtype=bool)
    apriori_flags[read_a_priori_int_flags(APRIORI_YAML_PATH, times=np.array(jds)).astype(int)] = True
    for ap_str in ap_strs:
        fh.update(ap_str, apriori_flags, apriori_flags, 'A Priori')
    fh.summarize('flagging a priori flagged times')
    apriori_flags |= solar_flags
else:
    apriori_flags = solar_flags

# STEP 2: FLAG TOTALLY DEAD ANTENNAS
for ap_str in ap_strs:
    fh.update(ap_str, metric_data['Dead?'][ap_str], metric_data['Dead?'][ap_str], 'Dead?')
fh.summarize('removing dead antennas')

# STEP 3: FLAG OUTRIGGERS
if SKIP_OUTRIGGERS:
    for ap_str in ap_strs:
        if int(ap_str[:-1]) >= 320:
            fh.update(ap_str, np.ones(Ncsvs, dtype=bool), np.ones(Ncsvs, dtype=bool), 'Outrigger')
    fh.summarize('flagging outriggers')

# STEP 4: FLAG ANTENNAS WITH INCONSISTENT RELABELING (see the report above)
for antnum in inconsistent:
    for ap_str in ap_strs:
        if int(ap_str[:-1]) == antnum:
            fh.update(ap_str, np.ones(Ncsvs, dtype=bool), np.ones(Ncsvs, dtype=bool), 'Inconsistent Relabeling')
fh.summarize('flagging antennas with temporally-inconsistent identity relabeling')

# STEP 5: FLAG CROSS-POLARIZED ANTENNAS
for ap_str in ap_strs:
    if np.mean(metric_data['Cross-Polarized'][ap_str]) < am_xpol_bad[1]:
        fh.update(ap_str, class_data['Cross-Polarized'][ap_str] == 'bad', np.ones(Ncsvs, dtype=bool), 'Cross-Polarized')
fh.summarize('removing cross-polarized antennas')

# STEP 6: FLAG POORLY-CORRELATING ANTENNAS
for ap_str in ap_strs:
    if np.mean(metric_data['Low Correlation'][ap_str]) < am_corr_bad[1]:
        fh.update(ap_str, class_data['Low Correlation'][ap_str] == 'bad', np.ones(Ncsvs, dtype=bool), 'Low Correlation')
fh.summarize('removing non-correlating antennas')

# STEP 7: FLAG ON AUTOCORRELATIONS
for category, ok_range, max_flag_frac in zip(['Autocorr Power', 'Autocorr Shape', 'Autocorr Slope', 'Auto RFI RMS'],
                                             [auto_power_suspect, auto_shape_good, auto_slope_good, auto_rfi_good],
                                             [AUTO_POWER_MAX_FLAG_FRAC, AUTO_SHAPE_MAX_FLAG_FRAC,
                                              AUTO_SLOPE_MAX_FLAG_FRAC, AUTO_RFI_MAX_FLAG_FRAC]):
    for ap_str in ap_strs:
        per_file_flags = (class_data[category][ap_str] == 'bad')
        # if not completely flagged or completely unflagged, grow flags between bad times
        if np.any(per_file_flags) and not np.all(fh.final_flags[ap_str] | per_file_flags):
            new_flags = grow_flags_between_bad_times(metric_data[category][ap_str], per_file_flags, ok_range)
            # if too many times are flagged for this category, flag the whole antenna (excluding sun-up times)
            if np.mean(new_flags[~apriori_flags]) > max_flag_frac:
                new_flags[:] = True
        else:
            new_flags = per_file_flags
        fh.update(ap_str, per_file_flags, new_flags, category)
    fh.summarize(f'flagging antennas for {category}')

# STEP 8: FLAG FOR HIGH SKY-MODEL CHI^2 (attributing packet issues to their own categories first)
for ap_str in ap_strs:
    flagged_for_cspa = (~fh.final_flags[ap_str]) & (class_data['Even/Odd Zeros'][ap_str] != 'bad') & \
                       (class_data['Bad Diff X-Engines'][ap_str] != 'bad') & (class_data['Sky Cal chi^2'][ap_str] == 'bad')
    if np.any(flagged_for_cspa) and not np.all(fh.final_flags[ap_str] | flagged_for_cspa):
        new_flags = grow_flags_between_bad_times(metric_data['Sky Cal chi^2'][ap_str], flagged_for_cspa, sc_cspa_suspect)
        if np.mean(new_flags[~apriori_flags]) > CHISQ_MAX_FLAG_FRAC:
            new_flags[:] = True
    else:
        new_flags = flagged_for_cspa
    fh.update(ap_str, flagged_for_cspa, new_flags, 'Sky Cal chi^2')
fh.summarize('flagging antennas for high sky-model chi^2')

# STEP 9: FLAG FOR CALIBRATED AUTOS INCONSISTENT WITH THE SKY MODEL
for ap_str in ap_strs:
    per_file_flags = (class_data['Cal Auto vs Model'][ap_str] == 'bad')
    if np.any(per_file_flags) and not np.all(fh.final_flags[ap_str] | per_file_flags):
        new_flags = grow_flags_between_bad_times(metric_data['Cal Auto vs Model'][ap_str], per_file_flags,
                                                 cal_auto_model_suspect)
        if np.mean(new_flags[~apriori_flags]) > CAL_AUTO_MAX_FLAG_FRAC:
            new_flags[:] = True
    else:
        new_flags = per_file_flags
    fh.update(ap_str, per_file_flags, new_flags, 'Cal Auto vs Model')
fh.summarize('flagging antennas whose calibrated autos are inconsistent with the sky model')

# STEP 10: FLAG FOR SNAP DECOHERENCE (per-SNAP metrics broadcast to each SNAP's antennas; unmeasured
# files are nan, so they neither certify recovery nor count as bad). Flags grow only through
# stretches where the smoothed metric stays at the BAD level: suspect-level decoherence is
# corrected rather than flagged, and worst-over-blocks sits near the suspect bound anyway
# because edge blocks carry a ~3-5% systematic floor
for category, ok_range in [('Worst SNAP Decoherence', (0, DECO_MAX_BAD)),
                           ('Worst Decoherence Jump', (0, DECO_JUMP_BAD))]:
    for ap_str in ap_strs:
        per_file_flags = (class_data[category][ap_str] == 'bad')
        if np.any(per_file_flags) and not np.all(fh.final_flags[ap_str] | per_file_flags):
            new_flags = grow_flags_between_bad_times(metric_data[category][ap_str], per_file_flags, ok_range)
            if np.mean(new_flags[~apriori_flags]) > DECO_MAX_FLAG_FRAC:
                new_flags[:] = True
        else:
            new_flags = per_file_flags
        fh.update(ap_str, per_file_flags, new_flags, category)
    fh.summarize(f'flagging antennas for {category}')

# STEP 11: FLAG FOR LOW IDENTITY COHERENCE. The max flag fraction is deliberately low: identity
# confusion is not time-local (a signal path does not heal mid-night), so substantial identity-bad
# time distrusts the label for the whole day
for ap_str in ap_strs:
    per_file_flags = (class_data['Identity Coherence'][ap_str] == 'bad')
    if np.any(per_file_flags) and not np.all(fh.final_flags[ap_str] | per_file_flags):
        new_flags = grow_flags_between_bad_times(metric_data['Identity Coherence'][ap_str], per_file_flags,
                                                 (IDENTITY_COHERENCE_SUSPECT, 1))
        if np.mean(new_flags[~apriori_flags]) > IDENTITY_MAX_FLAG_FRAC:
            new_flags[:] = True
    else:
        new_flags = per_file_flags
    fh.update(ap_str, per_file_flags, new_flags, 'Identity Coherence')
fh.summarize('flagging antennas for low identity coherence')

# STEP 12: FLAG FOR EVEN/ODD ZEROS AND EXCESS DIFF POWER (USUALLY PACKET ISSUES)
for category in ['Even/Odd Zeros', 'Bad Diff X-Engines']:
    for ap_str in ap_strs:
        per_file_flags = (class_data[category][ap_str] == 'bad')
        new_flags = np.array(per_file_flags)
        if np.mean(per_file_flags[~apriori_flags]) > XENGINE_MAX_FLAG_FRAC:
            new_flags[:] = True
        fh.update(ap_str, per_file_flags, new_flags, category)
    fh.summarize(f'flagging antennas for {category}')

# STEP 13: FLAG ANTENNAS THAT ARE ALREADY LARGELY FLAGGED
for ap_str in ap_strs:
    new_flags = np.array(fh.final_flags[ap_str])
    impose_max_flag_gap(new_flags)
    if np.mean(new_flags[~apriori_flags]) > OVERALL_MAX_FLAG_FRAC:
        new_flags[:] = True
    fh.update(ap_str, fh.final_flags[ap_str], new_flags, 'Frequently Flagged')
fh.summarize('flagging frequently-flagged antennas')

## Plotting

In [ ]:
def flagging_board():
    cmap = matplotlib.colors.ListedColormap(['blue', 'black', 'red', 'orange'])
    to_plot = np.vstack([np.where(~fh.final_flags[ap_str] & ~original_flags[ap_str], 0,
                                  np.where(~fh.final_flags[ap_str] & original_flags[ap_str], -1,
                                           np.where(fh.final_flags[ap_str] & ~original_flags[ap_str], 2, 1))) for ap_str in ap_strs])

    fig, ax = plt.subplots(figsize=(14, len(ants) / 10), dpi=100)
    im = ax.imshow(to_plot, aspect='auto', interpolation='none', cmap=cmap, vmin=-1.5, vmax=2.5,
               extent=[frac_jds[0], frac_jds[-1], len(ants), 0])
    ax.set_xlabel(f'JD - {int(jds[0])}')
    ax.set_yticks(ticks=np.arange(.5, len(ants)+.5))
    ax.set_yticklabels(labels=[ant for ant in ants], fontsize=6)
    ax.set_ylabel('Antenna Number (East First, Then North)')
    ax.tick_params(right=True, labelright=True)
    cbar = plt.colorbar(im, ax=ax, location='top', aspect=40)
    cbar.set_ticks([-1, 0, 1, 2])
    cbar.set_ticklabels(['Per-File Flag Removed', 'No Flags', 'Flagged Per-File and Here', 'Flagged Here Only'])

    ax2 = ax.twiny()
    ax2.set_xlim(lst_hours[0], lst_hours[-1])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_xlabel('LST (hours)')
    plt.tight_layout()

# *Figure 1: Flag Summary vs. JD*

This figure summarizes the flagging harmonization performed in this notebook, showing which flags were added (or potentially removed). 

In [ ]:
flagging_board()

In [ ]:
def flag_frac_array_plot():
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=100, gridspec_kw={'width_ratios': [2, 1]})

    def flag_frac_panel(ax, antnums, radius=7, legend=False):

        ang_dict = {'e': (225, 405), 'n': (45, 225)}

        xpos = np.array([hd.antpos[antnum][0] for antnum in ants if antnum in antnums])
        ypos = np.array([hd.antpos[antnum][1] for antnum in ants if antnum in antnums])
        scatter = ax.scatter(xpos, ypos, c='w', s=0)
        for ap_str in ap_strs:
            antnum, pol = int(ap_str[:-1]), ap_str[-1]
            if antnum in antnums:
                ax.add_artist(matplotlib.patches.Wedge(tuple(hd.antpos[antnum][0:2]), radius, *ang_dict[pol], color='grey'))
                flag_frac = np.mean(fh.final_flags[ap_str][~solar_flags])
                if flag_frac > .05:
                    ax.add_artist(matplotlib.patches.Wedge(tuple(hd.antpos[antnum][0:2]), radius * np.sqrt(flag_frac), *ang_dict[pol], color='r'))
                ax.text(hd.antpos[antnum][0], hd.antpos[antnum][1], str(antnum), color='w',  va='center', ha='center', zorder=100)

        ax.axis('equal')
        ax.set_xlim([np.min(xpos) - radius * 2, np.max(xpos) + radius * 2])
        ax.set_ylim([np.min(ypos) - radius * 2, np.max(ypos) + radius * 2])
        ax.set_xlabel("East-West Position (meters)", size=12)
        ax.set_ylabel("North-South Position (meters)", size=12)

        if legend:
            legend_objs = []
            legend_labels = []

            legend_objs.append(matplotlib.lines.Line2D([0], [0], marker='o', color='w', markeredgecolor='grey', markerfacecolor='grey', markersize=15))
            unflagged_nights = lambda pol: np.sum([np.mean(~fh.final_flags[ap_str][~solar_flags]) for ap_str in ap_strs if ap_str[-1] == pol])
            legend_labels.append((' \u2571\n').join([f'{unflagged_nights(pol):.1f} unflagged {pol}-polarized\nantenna-nights.' for pol in ['e', 'n']]))

            legend_objs.append(matplotlib.lines.Line2D([0], [0], marker='o', color='w', markeredgecolor='red', markerfacecolor='red', markersize=15))
            unflagged_nights = lambda pol: np.sum([np.mean(fh.final_flags[ap_str][~solar_flags]) for ap_str in ap_strs if ap_str[-1] == pol])
            legend_labels.append((' \u2571\n').join([f'{unflagged_nights(pol):.1f} flagged {pol}-polarized\nantenna-nights.' for pol in ['e', 'n']]))        
            ax.legend(legend_objs, legend_labels, ncol=1, fontsize=12)

    flag_frac_panel(axes[0], [ant for ant in ants if ant < 320], radius=7)
    flag_frac_panel(axes[1], [ant for ant in ants if ant >= 320], radius=50, legend=True)
    plt.tight_layout()

# *Figure 2: Array Flag Fraction Summary*

Flagging fraction of nighttime data for each antpol. Top-left semicircles are North-South polarized antpols; bottom right semicircles are East-West polarized antpols. Flag fraction is proportional to red area of each semicircle. Left panel is core antennas, right panel is outriggers.

In [ ]:
flag_frac_array_plot()

In [ ]:
def flag_summary_vs_jd():
    fig, ax = plt.subplots(figsize=(14, 5), dpi=100)
    ax.plot(frac_jds, np.mean(np.vstack([fh.final_flags[ap_str] for ap_str in ap_strs if ap_str[-1] == 'e']), axis=0), '.', ms=2, label='EW-Polarized')
    ax.plot(frac_jds, np.mean(np.vstack([fh.final_flags[ap_str] for ap_str in ap_strs if ap_str[-1] == 'n']), axis=0), '.', ms=2, label='NS-Polarized')
    ax.plot(frac_jds, np.mean(np.vstack([fh.final_flags[ap_str] for ap_str in ap_strs]), axis=0), 'k.', ms=3, label='All Antpols')
    ax.legend()
    ax.set_xlabel(f'JD - {int(np.floor(jds[0]))}')
    ax.set_ylabel('Fraction of Antennas Flagged')

    ax2 = ax.twiny()
    xmin, xmax = ax.get_xlim()
    lst_per_jd = (lst_hours[-1] - lst_hours[0]) / (frac_jds[-1] - frac_jds[0])
    ax2.set_xlim(lst_hours[0] + (xmin - frac_jds[0]) * lst_per_jd,
                 lst_hours[0] + (xmax - frac_jds[0]) * lst_per_jd)
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_xlabel('LST (hours)')
    plt.tight_layout()

# *Figure 3: Flag Fraction vs. JD Summary*

This plot shows the fraction of the array that's flagged for any reason as a function of time, both overall and per-polarization.

In [ ]:
flag_summary_vs_jd()

In [ ]:
def class_to_int(c):
    return np.where(c == 'bad', 1.7, np.where(c=='suspect', 1, 0))

def per_antenna_flag_harmonization_plots():
    # JD computations
    djd = np.median(np.diff(jds))

    # loop over ant-pols
    for ap_str in ap_strs:
        # if there are new nighttime, not-apriori flags
        if np.sum(fh.final_flags[ap_str][~apriori_flags]) > np.sum(original_flags[ap_str][~apriori_flags]):
            # if the new flags aren't just because of the OVERALL_MAX_FLAG_FRAC
            if np.mean(original_flags[ap_str][~apriori_flags]) <= OVERALL_MAX_FLAG_FRAC:
                for aps, category in fh.history.keys():
                    if category == 'Frequently Flagged':
                        continue
                    if ap_str == aps:
                        previous_flags, per_file_flags, new_flags = fh.history[(aps, category)]
                        # if new flags were added for this reason
                        if np.sum(new_flags[~apriori_flags]) > np.sum(per_file_flags[~apriori_flags]):
                            fig, ax = plt.subplots(figsize=(14, 3), dpi=100)
                            
                            # plot 
                            ax.scatter(frac_jds, metric_data[category][ap_str], c=class_to_int(class_data[category][ap_str]), s=3,
                                        vmin=0, vmax=1.7, cmap='RdYlGn_r', label='Metric/Classification')
                            ax.plot(frac_jds, nan_aware_smooth(metric_data[category][ap_str]), 'k--', label='Smoothed Metric')
                            ax.set_ylabel(category)
                            ax.set_xlabel(f'JD - {int(np.floor(jds[0]))}')
                            ax.set_xlim([np.min(frac_jds) - 10 * djd, 1.2 * np.max(frac_jds) - .2 * np.min(frac_jds)])
                            
                            # Indicate flagged stretches 
                            for i, bad_stretch in enumerate(true_stretches(per_file_flags)):
                                ax.axvspan(frac_jds[bad_stretch.start] - djd / 2, frac_jds[bad_stretch.stop - 1] + djd / 2, zorder=0, color='red', alpha=.75, lw=0,
                                            label=(f'Per-File Flags:\n{np.mean(per_file_flags[~solar_flags]):.2%} of night' if i == 0 else None))
                            for i, bad_stretch in enumerate(true_stretches(new_flags & ~apriori_flags)):
                                ax.axvspan(frac_jds[bad_stretch.start] - djd / 2, frac_jds[bad_stretch.stop - 1] + djd / 2, zorder=0, color='orange', alpha=.75, lw=0,
                                            label=(f'Harmonized Flags:\n{np.mean((new_flags & ~apriori_flags)[~solar_flags]):.2%} of night' if i == 0 else None))
                            for i, bad_stretch in enumerate(true_stretches((fh.final_flags[ap_str] & ~new_flags) | apriori_flags)):
                                ax.axvspan(frac_jds[bad_stretch.start] - djd / 2, frac_jds[bad_stretch.stop - 1] + djd / 2, zorder=0, color='purple', alpha=.75, lw=0,
                                            label=(f'All Final Flags:\n{np.mean(fh.final_flags[ap_str][~solar_flags]):.2%} of night' if i == 0 else None))                            

                            ax2 = ax.twiny()
                            xmin, xmax = ax.get_xlim()
                            lst_per_jd = (lst_hours[-1] - lst_hours[0]) / (frac_jds[-1] - frac_jds[0])
                            ax2.set_xlim(lst_hours[0] + (xmin - frac_jds[0]) * lst_per_jd,
                                         lst_hours[0] + (xmax - frac_jds[0]) * lst_per_jd)
                            mod24 = lambda x, _: f"{x % 24:.1f}"
                            ax2.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
                            ax2.set_xlabel('LST (hours)')

                            ax.legend(title=f'{ap_str}: {category}', loc='upper right')
                            plt.tight_layout()
                            plt.show()

# *Figure 4: Per-Antenna Flag Harmonization Summary*

This figure shows antennas that had their flags non-trivially modified by this notebook and tries to show the underlying rationale for why that happened. Sometimes the flag harmonizaton performed here leads to the whole antenna getting flagged; sometimes it just leads to large chunks of the night getting flagged. 

In [ ]:
per_antenna_flag_harmonization_plots()

## Save results

In [ ]:
add_to_history = 'Produced by full_day_antenna_flagging notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65

In [ ]:
if SAVE_RESULTS:
    out_flag_files = [cal.replace(SKY_CAL_SUFFIX, ANTENNA_FLAGS_SUFFIX) for cal in cal_files]
    for i, (cal, out_flag_file) in enumerate(zip(cal_files, out_flag_files)):
        # create UVFlag object based on UVCal
        uvc = UVCal()
        uvc.read_calfits(cal)
        uvf = UVFlag(uvc, mode='flag')
    
        # fill with flags
        for ant_ind, antnum in enumerate(uvf.ant_array):
            for pol_ind, polnum in enumerate(uvf.polarization_array):
                pol = {'Jee': 'e', 'Jnn': 'n'}[utils.jnum2str(polnum, x_orientation=uvf.telescope.get_x_orientation_from_feeds())]
                uvf.flag_array[ant_ind, :, :, pol_ind] = fh.final_flags[f'{antnum}{pol}'][i]

        # write to disk
        uvf.history += add_to_history        
        uvf.write(out_flag_file, clobber=True)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')